In [2]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os
import json

load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")
client = Anthropic(api_key=api_key)
model = "claude-haiku-4-5-20251001"

def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "content": text
    })

def add_assistant_message(messages, text):
    messages.append({
        "role": "assistant",
        "content": text
    })

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    res = client.messages.create(**params)
    return res.content[0].text



In [3]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for MongoDB-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text.strip())

In [4]:
dataset = generate_dataset()
print(dataset)

with open("dataset.json",'w') as f:
    json.dump(dataset, f, indent=2)

[{'task': 'Write a Python function that validates if a given string is a valid MongoDB ObjectId format (24 hexadecimal characters)'}, {'task': 'Create a JSON schema that defines a MongoDB document for a user collection with fields: name (string), email (string), age (integer), and createdAt (date)'}, {'task': 'Write a regex pattern that matches valid MongoDB collection names (alphanumeric characters, underscores, and dots, but cannot start with a dot or contain null characters)'}]


In [6]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert MongoDB code reviewer. Your task is to evaluate the following AI-generated solution.Make sure that code you are reviwing is as optimal as possible and as understandable as possible. You will be graded on the quality of your evaluation, so make sure to be thorough and provide detailed feedback.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)


def run_prompt(test_case):
    prompt = f"Solve the following tasks: {test_case['task']}"
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }

from statistics import mean
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score =mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

with open("dataset.json", "r") as f:
    dataset = json.load(f)

result = run_eval(dataset)

print(json.dumps(result, indent=2))

Average score: 6.666666666666667
[
  {
    "output": "# MongoDB ObjectId Validator\n\nHere's a comprehensive solution with multiple approaches:\n\n## Solution 1: Using Regular Expression (Recommended)\n\n```python\nimport re\n\ndef is_valid_mongodb_objectid(object_id: str) -> bool:\n    \"\"\"\n    Validates if a string is a valid MongoDB ObjectId format.\n    \n    MongoDB ObjectId consists of 24 hexadecimal characters.\n    \n    Args:\n        object_id: String to validate\n        \n    Returns:\n        bool: True if valid ObjectId format, False otherwise\n    \"\"\"\n    if not isinstance(object_id, str):\n        return False\n    \n    # Pattern: exactly 24 hexadecimal characters\n    pattern = r'^[0-9a-fA-F]{24}$'\n    return bool(re.match(pattern, object_id))\n\n\n# Test cases\ntest_cases = [\n    (\"507f1f77bcf86cd799439011\", True),      # Valid\n    (\"507F1F77BCF86CD799439011\", True),      # Valid (uppercase)\n    (\"507f1f77bcf86cd79943901\", False),      # Too short (2